# 실습 09 · FDA 데이터 무결성 & 감사추적 이상탐지
### 규제대응(FDA/GMP)에 AI 를 활용하는 실전 예제

**왜 중요한가 (현장 맥락)**
FDA·규제기관은 **데이터 무결성(Data Integrity)** 을 매우 중시합니다 (ALCOA+ 원칙).
감사추적(**audit trail**) 로그에는 누가·언제·무엇을 바꿨는지가 남는데,
데이터가 방대해 사람이 다 보기 어렵습니다.
**AI 이상탐지**로 의심스러운 활동(비정상 시각 수정, 반복 재시험, 권한 밖 변경)을
자동으로 찾아내면 규제대응·자체점검이 훨씬 효율적입니다.

**ALCOA+**: Attributable, Legible, Contemporaneous, Original, Accurate (+Complete, Consistent, Enduring, Available)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(3)

# ---- 실험실 시스템(LIMS/CDS) 감사추적 로그 생성 ----
n = 3000
users = ["김분석","이QC","박연구","최관리","정임시"]
actions = ["결과입력","결과수정","재시험","메소드변경","로그인","승인"]

# 정상: 업무시간(9~18시)에 일반적 활동
hours = np.random.choice(range(9,18), n, p=np.array([2,3,3,2,1,3,3,2,1])/20)
log = pd.DataFrame({
    "시각_시": hours,
    "요일": np.random.choice(range(5), n),           # 0~4 평일
    "사용자": np.random.choice(users, n),
    "행위": np.random.choice(actions, n, p=[.35,.10,.05,.03,.35,.12]),
    "수정횟수": np.random.poisson(0.2, n),           # 한 결과를 몇 번 고쳤나
    "결과값_변화율": np.abs(np.random.normal(0,0.02,n)),  # 수정 시 값이 얼마나 바뀜
})

# ---- 의심 활동 주입 (데이터 무결성 위반 시나리오) ----
susp = []
# 심야에 결과 수정 (contemporaneous 위반)
for i in np.random.choice(n, 15, replace=False):
    log.loc[i,["시각_시","행위","수정횟수","결과값_변화율"]] = [2, "결과수정", 4, 0.25]; susp.append(i)
# 규격 통과시키려 반복 재시험(testing into compliance)
for i in np.random.choice(n, 12, replace=False):
    log.loc[i,["행위","수정횟수","결과값_변화율"]] = ["재시험", 6, 0.35]; susp.append(i)
log["실제의심"] = 0; log.loc[susp,"실제의심"]=1
print("전체 로그:", n, "| 주입한 의심 활동:", len(set(susp)))
log.head()

## 1. 규칙 기반 점검 (전통적 방법)
먼저 명시적 규칙으로 걸러봅니다. (규제 현장에서 실제 쓰는 1차 필터)


In [ ]:
rule_flag = (
    (log["시각_시"] < 6) |            # 심야 활동
    (log["수정횟수"] >= 3) |          # 과도한 수정
    (log["결과값_변화율"] > 0.1)      # 값이 크게 바뀐 수정
)
print("규칙 기반으로 걸린 로그 수:", rule_flag.sum())
# 규칙은 단순명료하나, '조합적으로 이상한' 패턴이나 새로운 수법은 놓칠 수 있음

## 2. ⭐ AI 다변량 이상탐지
행위·시각·수정패턴을 **함께** 보고 정상 패턴에서 벗어난 로그를 점수화합니다.
범주형(사용자·행위)은 원-핫 인코딩으로 숫자화합니다.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 특성 만들기 (범주형 → 숫자)
feat = pd.get_dummies(log[["시각_시","요일","사용자","행위","수정횟수","결과값_변화율"]],
                      columns=["사용자","행위"])
Xs = StandardScaler().fit_transform(feat)

iso = IsolationForest(contamination=0.02, random_state=42).fit(Xs)
log["이상점수"] = -iso.score_samples(Xs)
log["AI탐지"] = (iso.predict(Xs)==-1).astype(int)
print("AI 가 탐지한 이상 로그:", log["AI탐지"].sum(), "건")

In [ ]:
# 탐지 성능 확인 — 주입한 '실제 의심'을 얼마나 잡아냈나
from sklearn.metrics import classification_report, roc_auc_score
print(classification_report(log["실제의심"], log["AI탐지"],
                            target_names=["정상","의심"]))
print("이상점수 기준 ROC-AUC:", round(roc_auc_score(log["실제의심"], log["이상점수"]),3))

In [ ]:
# 이상점수 상위 로그를 우선 검토 리스트로 (감사관/QA 가 집중 조사)
cols = ["시각_시","사용자","행위","수정횟수","결과값_변화율","이상점수"]
log.sort_values("이상점수", ascending=False)[cols].head(12).round(3)

In [ ]:
# 시각화: 시각 vs 수정횟수 (심야·과다수정 영역에 의심 활동이 모임)
plt.figure(figsize=(7,4.5))
normal = log[log["AI탐지"]==0]; anom = log[log["AI탐지"]==1]
plt.scatter(normal["시각_시"], normal["수정횟수"], alpha=0.2, label="정상")
plt.scatter(anom["시각_시"], anom["수정횟수"], color="red", label="AI 이상탐지")
plt.xlabel("행위 시각(시)"); plt.ylabel("수정 횟수"); plt.legend()
plt.title("감사추적 이상탐지 (심야·과다수정 활동 포착)"); plt.show()

## 3. 규칙 기반 vs AI — 함께 쓰기
- **규칙**: 명확·설명 쉬움·규제 근거 제시 용이 → 1차 필터
- **AI**: 미지의 패턴·조합적 이상 포착 → 2차 심화
- 실무에서는 둘을 **병행**하고, AI 가 표시한 항목은 **사람이 최종 판단**(human-in-the-loop)

**AI 검증(GAMP 5) 관점**: 규제환경에서 AI 를 쓰려면 모델의 목적·데이터·검증기록을
문서화해야 합니다. LLM 에게 "이 이상탐지 결과를 QA 리포트 형식으로 요약해줘"라고 하면
보고서 초안 작성도 자동화할 수 있습니다.


## 정리 & 현장 응용
- 감사추적 로그에서 **데이터 무결성 위반 의심 활동**을 AI 로 자동 선별
- 규칙 기반(1차) + AI 이상탐지(2차) + 사람 판단(최종)의 3단 구조
- 제약 적용: LIMS/CDS 감사추적, 전자배치기록(EBR), 접근권한 로그, OOS 조사
- ALCOA+ 원칙과 연결해 **규제대응 자체점검** 도구로 활용
